# Reproducibility package

**Quantifying the Information Capacity of Histone Post-Translational
Modifications in Human Chromatin**

This standalone notebook regenerates every numerical value and numeric table
in the manuscript from closed-form Shannon-entropy expressions. It needs no
external data or network access, runs top to bottom in a few seconds, and
writes no files: all results, including Word-pasteable tables, render inline.

The modules follow the manuscript section by section, so the notebook can be
read and run side by side with the text. Every parameter lives in the single
setup cell; editing it adapts the whole framework, and the per-module value
checks are then skipped automatically (see `RUN_REGRESSION_CHECKS`).

| Module | Manuscript target |
|--------|-------------------|
| 1 | Theoretical capacity and accessible proteoform space (Results 2.1-2.2; Table 1) |
| 2 | Core-histone PTM information hierarchy (Results 2.2-2.3; Table 2) |
| 3 | Linker histone H1 and integrated estimates (Results 2.4; Table 3) |
| 4 | Model parameters (Methods; Table M1) |

**Cite:** Zenodo `10.5281/zenodo.21110251`.

## Setup: environment check, parameters, and helpers

One cell holding the single source of truth for the parameters, the
information-theoretic functions, and the inline table renderer.

In [1]:
import importlib.metadata as _meta
import math
import platform
import re
import sys

import pandas as pd

MIN_PYTHON = (3, 10)
REQUIRED = {"pandas": (1, 4), "jinja2": (3, 1, 2)}


def _ver_tuple(text, n):
    """Leading numeric components of a version string as an n-tuple."""
    parts = [int(x) for x in re.findall(r"\d+", text)[:n]]
    return tuple(parts + [0] * (n - len(parts)))


def _check_environment():
    """Validate the interpreter and dependencies; print status or raise."""
    if sys.version_info < MIN_PYTHON:
        raise RuntimeError(
            f"Python >= {'.'.join(map(str, MIN_PYTHON))} required, "
            f"found {platform.python_version()}"
        )
    rows = [("Python", platform.python_version())]
    for pkg, min_ver in REQUIRED.items():
        try:
            found = _meta.version(pkg)
        except _meta.PackageNotFoundError as exc:
            need = ".".join(map(str, min_ver))
            raise RuntimeError(f"{pkg} >= {need} required; not installed") \
                from exc
        if _ver_tuple(found, len(min_ver)) < min_ver:
            need = ".".join(map(str, min_ver))
            raise RuntimeError(f"{pkg} >= {need} required, found {found}")
        rows.append((pkg, found))
    width = max(len(name) for name, _ in rows)
    for name, ver in rows:
        print(f"  OK  {name:<{width}}  {ver}")
    print("\nAll requirements satisfied. Ready to run.")


_check_environment()


# --- Parameters (edit to adapt the framework) ---------------------------
GENOME_BP = 6.4e9           # diploid human genome size (bp)
NRL = 200                   # average nucleosome repeat length (bp)
N_CORE = 30                 # effective core-histone PTM-bearing sites
S = 5                       # mutually exclusive biochemical states per site
K_ACCESSIBLE = (1e3, 1e4)   # accessible nucleosomal proteoform space
K_STAT = (1e2, 1e3)         # dominant occupied core PTM states
ALPHA = (0.5, 0.75)         # correlation/redundancy correction factor
N_FUNC = (8, 16)            # distinguishable functional chromatin states
N_H1_RESIDUES = 400         # potential H1 PTM-bearing residues (extreme)
N_H1_EFF = (10, 15)         # effective H1 PTM-bearing sites
K_H1_STAT = (1e2, 1e3)      # dominant occupied H1 PTM states
RHO = (0.5, 1.0)            # H1 molecules per nucleosome (occupancy)
H1_FUNC_BITS = (1, 2)       # functionally interpretable H1 information (bits)

# Derived whole-cell scaling factor (Methods 2): nucleosomes per cell.
N_NUC = GENOME_BP / NRL

# --- Published defaults and regression-check gate -----------------------
# The per-module assertions pin the published values as a reproduction
# guarantee. They must not fire when the parameters above are changed for
# adaptation, so they run only while every parameter matches the defaults.
_PUBLISHED_DEFAULTS = {
    "GENOME_BP": 6.4e9, "NRL": 200, "N_CORE": 30, "S": 5,
    "K_ACCESSIBLE": (1e3, 1e4), "K_STAT": (1e2, 1e3),
    "ALPHA": (0.5, 0.75), "N_FUNC": (8, 16), "N_H1_RESIDUES": 400,
    "N_H1_EFF": (10, 15), "K_H1_STAT": (1e2, 1e3), "RHO": (0.5, 1.0),
    "H1_FUNC_BITS": (1, 2),
}
RUN_REGRESSION_CHECKS = {
    "GENOME_BP": GENOME_BP, "NRL": NRL, "N_CORE": N_CORE, "S": S,
    "K_ACCESSIBLE": K_ACCESSIBLE, "K_STAT": K_STAT, "ALPHA": ALPHA,
    "N_FUNC": N_FUNC, "N_H1_RESIDUES": N_H1_RESIDUES, "N_H1_EFF": N_H1_EFF,
    "K_H1_STAT": K_H1_STAT, "RHO": RHO, "H1_FUNC_BITS": H1_FUNC_BITS,
} == _PUBLISHED_DEFAULTS
if not RUN_REGRESSION_CHECKS:
    print("Parameters differ from published defaults; "
          "regression checks skipped.")


# --- Information-theoretic helpers and table renderer -------------------
def entropy_uniform(num_states):
    """Shannon entropy of `num_states` equiprobable outcomes, log2(K)."""
    return math.log2(num_states)


def max_entropy(n_sites, n_states):
    """Upper-bound entropy of `n_sites` independent sites with `n_states`
    equiprobable states each, N × log2(S) bits."""
    return n_sites * math.log2(n_states)


def bits_to_mb(total_bits):
    """Information in bits to megabytes (1 byte = 8 bits, 1 MB = 1e6 B)."""
    return total_bits / 8 / 1e6


def bits_to_gb(total_bits):
    """Information in bits to gigabytes (1 GB = 1e9 bytes)."""
    return total_bits / 8 / 1e9


def _as_range(value):
    """Coerce a scalar to a degenerate (low, high) range; pass ranges on.

    The framework propagates lower/upper bounds as 2-tuples so that a scalar
    (e.g. the single theoretical capacity) and a range compose under one set
    of operators."""
    return value if isinstance(value, tuple) else (value, value)


def map_range(func, rng):
    """Apply a monotonic increasing `func` to each end of a range."""
    return func(rng[0]), func(rng[1])


def mul_range(a, b):
    """Element-wise product of two positive ranges (matched endpoints)."""
    a, b = _as_range(a), _as_range(b)
    return a[0] * b[0], a[1] * b[1]


def add_range(a, b):
    """Element-wise sum of two ranges or scalars (matched endpoints)."""
    a, b = _as_range(a), _as_range(b)
    return a[0] + b[0], a[1] + b[1]


def scale_capacity(bits_value, n_units, to=bits_to_mb):
    """Whole-cell capacity for a bits value/range over a unit count
    value/range, pairing low-with-low and high-with-high endpoints.

    Endpoint pairing (not the full cross-product) matches the manuscript's
    convention of reporting min×min and max×max for monotonic ranges."""
    bits_value, n_units = _as_range(bits_value), _as_range(n_units)
    return (to(bits_value[0] * n_units[0]),
            to(bits_value[1] * n_units[1]))


def decimal(value, dec=1):
    """One scalar at `dec` decimal places, matching the manuscript's
    precision; magnitudes >= 1e4 use universal 'm × 10^n' notation rather
    than Python's exponential form."""
    if value != 0 and abs(value) >= 1e4:
        exp = math.floor(math.log10(abs(value)))
        return f"{value / 10.0 ** exp:.{dec}f} × 10^{exp}"
    return f"{value:.{dec}f}"


def fmt(value, dec=1, unit=""):
    """Format a scalar or (low, high) range at `dec` decimal places; collapse
    equal endpoints and join ranges with an en-dash. A leading tilde matches
    the manuscript style. Display only; arithmetic stays full precision."""
    tail = f" {unit}" if unit else ""
    if isinstance(value, tuple):
        low, high = decimal(value[0], dec), decimal(value[1], dec)
        return f"~{low}{tail}" if low == high else f"~{low}–{high}{tail}"
    return f"~{decimal(value, dec)}{tail}"


def render_table(frame, caption):
    """Index-free, caption-bearing table; renders inline and pastes into
    Word as a native table."""
    return frame.style.hide(axis="index").set_caption(caption)

  OK  Python  3.13.14
  OK  pandas  3.0.1
  OK  jinja2  3.1.6

All requirements satisfied. Ready to run.


## Reproducibility Module 1. Theoretical capacity and accessible proteoform space (Table 1; Results 2.1 to 2.2)

This module builds Table 1 (core nucleosome PTM information capacity
estimates), whose rows are the theoretical upper bound from Results 2.1 and
the accessible proteoform space from Results 2.2. It computes the
combinatorial proteoform space $\Omega = S^{N_{\mathrm{core}}}$ and its
Shannon upper bound $H_{\mathrm{core,max}} = N_{\mathrm{core}} \times \log_2
S$, then multiplies $H_{\mathrm{core,max}}$ by the number of nucleosomes per
diploid human cell $N_{\mathrm{nuc}}$ to give the theoretical capacity
$I_{\mathrm{core,max}}$. It also computes the accessible-state entropy under
an equiprobable assumption, $H_{\mathrm{core,accessible}} = \log_2 K$ for $K
= 10^3$ to $10^4$, and its whole-cell value $I_{\mathrm{core,accessible}}$.
The module prints $\Omega$, $H_{\mathrm{core,max}}$, $N_{\mathrm{nuc}}$ and
$I_{\mathrm{core,max}}$, and renders Table 1.

In [2]:
omega = S ** N_CORE
h_core_max = max_entropy(N_CORE, S)
i_core_max_mb = scale_capacity(h_core_max, N_NUC)[0]
h_acc = map_range(entropy_uniform, K_ACCESSIBLE)
i_acc = scale_capacity(h_acc, N_NUC)

if RUN_REGRESSION_CHECKS:
    assert round(math.log2(S), 4) == 2.3219
    assert round(omega / 1e20, 1) == 9.3
    assert N_NUC == 3.2e7
    assert round(h_core_max, 1) == 69.7
    assert round(i_core_max_mb, 1) == 278.6
    assert tuple(round(x, 2) for x in h_acc) == (9.97, 13.29)
    assert tuple(round(x, 1) for x in i_acc) == (39.9, 53.2)

print(f"Ω          = S^N_core = {S}^{N_CORE} = {decimal(omega, 1)} states")
print(f"H_core,max = N_core × log2(S) = {decimal(h_core_max, 1)} bits")
print(f"N_nuc      = {decimal(GENOME_BP, 1)} bp / {NRL} bp "
      f"= {decimal(N_NUC, 1)} nucleosomes")
print(f"I_core,max = N_nuc × H_core,max = {decimal(i_core_max_mb, 1)} MB")

table1 = pd.DataFrame({
    "Estimate": [
        "Theoretical upper bound",
        "Accessible proteoform lower estimate",
        "Accessible proteoform upper estimate",
    ],
    "Entropy per nucleosome": [
        fmt(h_core_max, dec=1, unit="bits"),
        fmt(h_acc[0], dec=2, unit="bits"),
        fmt(h_acc[1], dec=2, unit="bits"),
    ],
    "Total capacity per diploid human cell": [
        fmt(i_core_max_mb, dec=1, unit="MB"),
        fmt(i_acc[0], dec=1, unit="MB"),
        fmt(i_acc[1], dec=1, unit="MB"),
    ],
    "Interpretation": [
        "Mathematical ceiling assuming 30 independent PTM sites and "
        "five possible states per site",
        "Approximation based on 10^3 accessible nucleosomal proteoforms",
        "Approximation based on 10^4 accessible nucleosomal proteoforms",
    ],
})
render_table(
    table1, "Table 1. Core nucleosome PTM information capacity estimates.")

Ω          = S^N_core = 5^30 = 9.3 × 10^20 states
H_core,max = N_core × log2(S) = 69.7 bits
N_nuc      = 6.4 × 10^9 bp / 200 bp = 3.2 × 10^7 nucleosomes
I_core,max = N_nuc × H_core,max = 278.6 MB


Estimate,Entropy per nucleosome,Total capacity per diploid human cell,Interpretation
Theoretical upper bound,~69.7 bits,~278.6 MB,Mathematical ceiling assuming 30 independent PTM sites and five possible states per site
Accessible proteoform lower estimate,~9.97 bits,~39.9 MB,Approximation based on 10^3 accessible nucleosomal proteoforms
Accessible proteoform upper estimate,~13.29 bits,~53.2 MB,Approximation based on 10^4 accessible nucleosomal proteoforms


## Reproducibility Module 2. Core-histone PTM information hierarchy (Table 2; Results 2.2 to 2.3)

This module builds Table 2 (the five-level hierarchy of core histone PTM
information estimates). Its theoretical and accessible rows are reused from
Module 1; the module adds the three remaining layers of the hierarchy. The
statistical occupancy-corrected layer (Results 2.2) evaluates the Shannon
form $H_{\mathrm{stat}} = -\sum_i p_i \log_2 p_i$, which reduces, under the
equiprobable approximation used in the main text, to $H_{\mathrm{core,stat}}
= \log_2 K_{\mathrm{eff}}$, with $K_{\mathrm{eff}} = 10^2$ to $10^3$. The
correlation-corrected layer (Results 2.3) applies the redundancy factor
$H_{\mathrm{core,corr}} = \alpha H_{\mathrm{stat}}$ with $\alpha = 0.5$ to
$0.75$; this parametric discount stands in for the joint-entropy
decomposition $H(X_1,\ldots,X_N) = \sum_i H(X_i) - \sum_{i<j} I(X_i;X_j) +
\cdots$, whose pairwise mutual-information terms cannot be evaluated without
joint distributions that are not currently available. The functional layer
$I_{\mathrm{func}}$ (Results 2.3) uses the mutual information $I(X;Y) =
\sum_{x,y} p(x,y) \log_2 \frac{p(x,y)}{p(x)p(y)}$ between the nucleosomal
PTM configuration X and the functional chromatin state Y; lacking an
empirical joint distribution, it is evaluated as the upper bound
$I_{\mathrm{core,func}} \le \log_2(8\ \text{to}\ 16)$, giving 3 to 4 bits.
Each added layer is scaled to the whole cell ($I_{\mathrm{core,stat}}$,
$I_{\mathrm{core,corr}}$ and $I_{\mathrm{core,func,total}}$), and the module
renders Table 2.

In [3]:
h_stat = map_range(entropy_uniform, K_STAT)
h_corr = mul_range(ALPHA, h_stat)
h_func = map_range(entropy_uniform, N_FUNC)

i_stat = scale_capacity(h_stat, N_NUC)
i_corr = scale_capacity(h_corr, N_NUC)
i_func = scale_capacity(h_func, N_NUC)

if RUN_REGRESSION_CHECKS:
    assert tuple(round(x, 2) for x in h_stat) == (6.64, 9.97)
    assert tuple(round(x, 2) for x in h_corr) == (3.32, 7.47)
    assert tuple(round(x, 1) for x in h_func) == (3.0, 4.0)
    assert tuple(round(x, 1) for x in i_stat) == (26.6, 39.9)
    assert tuple(round(x, 1) for x in i_corr) == (13.3, 29.9)
    assert tuple(round(x, 1) for x in i_func) == (12.0, 16.0)

table2 = pd.DataFrame({
    "Estimate": [
        "Theoretical combinatorial capacity",
        "Accessible proteoform space",
        "Statistical occupancy-corrected entropy",
        "Correlation-corrected entropy",
        "Functional chromatin-state information",
    ],
    "Entropy per nucleosome": [
        fmt(h_core_max, dec=1, unit="bits"),
        fmt(h_acc, dec=2, unit="bits"),
        fmt(h_stat, dec=2, unit="bits"),
        fmt(h_corr, dec=2, unit="bits"),
        fmt(h_func, dec=0, unit="bits"),
    ],
    "Total capacity per diploid human cell": [
        fmt(i_core_max_mb, dec=1, unit="MB"),
        fmt(i_acc, dec=1, unit="MB"),
        fmt(i_stat, dec=1, unit="MB"),
        fmt(i_corr, dec=1, unit="MB"),
        fmt(i_func, dec=0, unit="MB"),
    ],
    "Biological meaning": [
        "All modeled independent PTM combinations",
        "Biologically accessible PTM proteoform space",
        "Dominant occupied states after non-uniformity is considered",
        "Independent information after redundancy and biochemical coupling",
        "Regulatory information interpretable by cellular machinery",
    ],
})
render_table(
    table2,
    "Table 2. Hierarchy of core histone PTM information estimates and "
    "biological interpretation.",
)

Estimate,Entropy per nucleosome,Total capacity per diploid human cell,Biological meaning
Theoretical combinatorial capacity,~69.7 bits,~278.6 MB,All modeled independent PTM combinations
Accessible proteoform space,~9.97–13.29 bits,~39.9–53.2 MB,Biologically accessible PTM proteoform space
Statistical occupancy-corrected entropy,~6.64–9.97 bits,~26.6–39.9 MB,Dominant occupied states after non-uniformity is considered
Correlation-corrected entropy,~3.32–7.47 bits,~13.3–29.9 MB,Independent information after redundancy and biochemical coupling
Functional chromatin-state information,~3–4 bits,~12–16 MB,Regulatory information interpretable by cellular machinery


## Reproducibility Module 3. Linker histone H1 and integrated estimates (Table 3; Results 2.4)

This module builds Table 3 (contribution of linker histone H1), covering
Results 2.4. It reproduces the H1 layer and the integrated core-plus-H1
estimates across the same hierarchy as the core histones. The H1 occupancy
$\rho = 0.5$ to $1.0$ gives $N_{\mathrm{H1}} = \rho \times N_{\mathrm{nuc}}$
molecules per cell. The extreme ceiling treats all of the roughly 400
potential H1 residues as independent sites, $H_{\mathrm{H1,max}} = 400
\log_2 S$, reported in gigabytes as $I_{\mathrm{H1,max}}$; the constrained
estimate uses $N_{\mathrm{H1,eff}} = 10$ to $15$ effective sites,
$H_{\mathrm{H1,eff}} = N_{\mathrm{H1,eff}} \log_2 S$, giving
$I_{\mathrm{H1,eff}}$ at the whole-cell scale. The statistical and
correlation-corrected H1 layers reuse the assumptions from the core
histones, $H_{\mathrm{H1,stat}} = \log_2 K_{\mathrm{eff}}$ and
$H_{\mathrm{H1,corr}} = \alpha H_{\mathrm{H1,stat}}$, and the functional H1
contribution is taken as 1 to 2 bits per molecule ($I_{\mathrm{H1,func}}$).
Core and H1 estimates are summed at each hierarchy level,
$I_{\mathrm{core+H1}} = I_{\mathrm{core}} + I_{\mathrm{H1}}$, to give
$I_{\mathrm{core+H1,max}}$, $I_{\mathrm{core+H1,eff}}$,
$I_{\mathrm{core+H1,stat}}$, $I_{\mathrm{core+H1,corr}}$ and
$I_{\mathrm{core+H1,func}}$, reusing the theoretical core capacity
$I_{\mathrm{core,max}}$ from Module 1 in the extreme (max) and effective
(eff) rows. The module renders Table 3.

In [4]:
n_h1 = map_range(lambda r: r * N_NUC, RHO)            # H1 molecules per cell

# H1 entropy per molecule (bits) at each level.
h_h1_max = max_entropy(N_H1_RESIDUES, S)
h_h1_eff = map_range(lambda n: max_entropy(n, S), N_H1_EFF)
h_h1_stat = map_range(entropy_uniform, K_H1_STAT)
h_h1_corr = mul_range(ALPHA, h_h1_stat)

# H1 whole-cell capacity (extreme limit in GB, the rest in MB).
i_h1_max = scale_capacity(h_h1_max, n_h1, to=bits_to_gb)
i_h1_eff = scale_capacity(h_h1_eff, n_h1)
i_h1_stat = scale_capacity(h_h1_stat, n_h1)
i_h1_corr = scale_capacity(h_h1_corr, n_h1)
i_h1_func = scale_capacity(H1_FUNC_BITS, n_h1)

# Matching core estimates (theoretical reused for extreme/effective rows).
i_core_max_gb = scale_capacity(h_core_max, N_NUC, to=bits_to_gb)[0]

# Integrated core + H1 totals (element-wise endpoint sums).
i_tot_max = add_range(i_core_max_gb, i_h1_max)
i_tot_eff = add_range(i_core_max_mb, i_h1_eff)
i_tot_stat = add_range(i_stat, i_h1_stat)
i_tot_corr = add_range(i_corr, i_h1_corr)
i_tot_func = add_range(i_func, i_h1_func)

if RUN_REGRESSION_CHECKS:
    assert round(h_h1_max, 1) == 928.8
    assert tuple(round(x, 1) for x in h_h1_eff) == (23.2, 34.8)
    assert tuple(round(x, 2) for x in i_h1_max) == (1.86, 3.72)
    assert tuple(round(x, 1) for x in i_h1_eff) == (46.4, 139.3)
    assert tuple(round(x, 1) for x in i_h1_stat) == (13.3, 39.9)
    assert tuple(round(x, 1) for x in i_h1_corr) == (6.6, 29.9)
    assert tuple(round(x, 1) for x in i_h1_func) == (2.0, 8.0)
    assert tuple(round(x, 2) for x in i_tot_max) == (2.14, 3.99)
    assert tuple(round(x, 1) for x in i_tot_eff) == (325.1, 417.9)
    assert tuple(round(x, 1) for x in i_tot_stat) == (39.9, 79.7)
    assert tuple(round(x, 1) for x in i_tot_corr) == (19.9, 59.8)
    assert tuple(round(x, 1) for x in i_tot_func) == (14.0, 24.0)

table3 = pd.DataFrame({
    "Estimate": [
        "Extreme theoretical H1 limit",
        "Effective H1 upper bound",
        "Statistical entropy estimate",
        "Correlation-corrected estimate",
        "Functional interpretable estimate",
    ],
    "Core histones only": [
        fmt(i_core_max_mb, dec=1, unit="MB"),
        fmt(i_core_max_mb, dec=1, unit="MB"),
        fmt(i_stat, dec=1, unit="MB"),
        fmt(i_corr, dec=1, unit="MB"),
        fmt(i_func, dec=0, unit="MB"),
    ],
    "H1 contribution": [
        fmt(i_h1_max, dec=2, unit="GB"),
        fmt(i_h1_eff, dec=1, unit="MB"),
        fmt(i_h1_stat, dec=1, unit="MB"),
        fmt(i_h1_corr, dec=1, unit="MB"),
        fmt(i_h1_func, dec=0, unit="MB"),
    ],
    "Core + H1 total": [
        fmt(i_tot_max, dec=2, unit="GB"),
        fmt(i_tot_eff, dec=1, unit="MB"),
        fmt(i_tot_stat, dec=1, unit="MB"),
        fmt(i_tot_corr, dec=1, unit="MB"),
        fmt(i_tot_func, dec=0, unit="MB"),
    ],
    "Interpretation": [
        "Mathematical ceiling; not biologically realistic",
        "Constrained estimate using 10-15 effective H1 PTM sites",
        "Occupancy-weighted biological state space",
        "Independent information after biochemical redundancy and coupling",
        "Regulatory information interpretable by cellular machinery",
    ],
})
render_table(
    table3, "Table 3. Contribution of linker histone H1 to chromatin PTM "
            "information estimates.")

Estimate,Core histones only,H1 contribution,Core + H1 total,Interpretation
Extreme theoretical H1 limit,~278.6 MB,~1.86–3.72 GB,~2.14–3.99 GB,Mathematical ceiling; not biologically realistic
Effective H1 upper bound,~278.6 MB,~46.4–139.3 MB,~325.1–417.9 MB,Constrained estimate using 10-15 effective H1 PTM sites
Statistical entropy estimate,~26.6–39.9 MB,~13.3–39.9 MB,~39.9–79.7 MB,Occupancy-weighted biological state space
Correlation-corrected estimate,~13.3–29.9 MB,~6.6–29.9 MB,~19.9–59.8 MB,Independent information after biochemical redundancy and coupling
Functional interpretable estimate,~12–16 MB,~2–8 MB,~14–24 MB,Regulatory information interpretable by cellular machinery


## Reproducibility Module 4. Model parameters (Table M1; Methods)

This module builds Table M1 (the main parameters of the framework, presented
in the Methods section). It assembles the table directly from the setup-cell
parameters, so that the reported values are guaranteed to match those used
in the calculations.

In [5]:
def _pow10(rng):
    """Render a power-of-ten range as '10^a-10^b'."""
    lo, hi = (int(round(math.log10(v))) for v in rng)
    return f"10^{lo}-10^{hi} states"


table_m1 = pd.DataFrame({
    "Parameter": [
        "N_nuc", "N_core", "S", "K_accessible", "K_stat",
        "alpha", "rho", "N_H1,eff",
    ],
    "Value or range": [
        fmt(N_NUC, dec=1, unit="nucleosomes"),
        f"{N_CORE:d} effective sites",
        f"{S:d} states per site",
        _pow10(K_ACCESSIBLE),
        _pow10(K_STAT),
        f"{ALPHA[0]:.2g}-{ALPHA[1]:.2g}",
        f"{RHO[0]:.1f}-{RHO[1]:.1f} H1 molecules per nucleosome",
        f"{N_H1_EFF[0]:d}-{N_H1_EFF[1]:d} effective sites",
    ],
    "Purpose": [
        "Scaling factor for whole-cell estimates",
        "Conservative number of core-histone PTM variables",
        "Unmodified, acetylated, methylated, phosphorylated or "
        "ubiquitinated",
        "Accessible nucleosomal proteoform space",
        "Dominant occupied nucleosomal PTM states",
        "Correlation/redundancy correction factor",
        "H1 occupancy parameter",
        "Constrained H1 PTM-variable estimate",
    ],
})
render_table(table_m1, "Table M1. Main parameters used in the "
                       "information-theoretic framework.")

Parameter,Value or range,Purpose
N_nuc,~3.2 × 10^7 nucleosomes,Scaling factor for whole-cell estimates
N_core,30 effective sites,Conservative number of core-histone PTM variables
S,5 states per site,"Unmodified, acetylated, methylated, phosphorylated or ubiquitinated"
K_accessible,10^3-10^4 states,Accessible nucleosomal proteoform space
K_stat,10^2-10^3 states,Dominant occupied nucleosomal PTM states
alpha,0.5-0.75,Correlation/redundancy correction factor
rho,0.5-1.0 H1 molecules per nucleosome,H1 occupancy parameter
"N_H1,eff",10-15 effective sites,Constrained H1 PTM-variable estimate
